# Transformer-Based Text Models for Endodontic Treatment Outcome Prediction

This notebook contains all text-based transformer experiments for predicting endodontic treatment outcomes from clinical text. Five model configurations are compared.

## 1. Setup & Configuration

In [ ]:
import os, json, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from datasets import load_dataset, Value
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
from sklearn.model_selection import StratifiedGroupKFold
from datasets import Dataset
from sklearn.metrics import (
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# ── Global config ────────────────────────────────────────────────────────
OUTER_FOLDS = 5
INNER_FOLDS = 5
PATIENT_COL = "patient_id"  # REQUIRED: replace with actual patient identifier column
BOOTSTRAP_SAMPLES = 2000
RANDOM_STATE = 42
DATA_DIR     = "data"
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 2. Data Preparation for patient-grouped nested cross-validation

No single train/validation/test split is used. Patient identifiers are mandatory. Each outer fold is held out from inner model selection and threshold optimization. The patient-level dataset is not distributed.


In [ ]:
TEXT_COL_EN     = "prognosis_english"    # translated prognosis note
TEXT_COL_DANISH = "prognosis_danish"     # original Danish text — update if column name differs
LABEL_COL       = "is_effective_romexis"

# Columns concatenated for the "prognosis + signs & symptoms" experiments.
# Update to match the exact column names in your dataset.
ALL_TEXT_COLS   = ["prognosis_english"]  # REQUIRED: add actual signs/symptoms columns before running all-text experiments

In [ ]:
path = os.path.join(DATA_DIR, "Erda, final, anonymized,for LLM and multimodal.csv")
df = pd.read_csv(path)
required = [PATIENT_COL, LABEL_COL, TEXT_COL_EN, TEXT_COL_DANISH] + ALL_TEXT_COLS
missing = sorted(set(required) - set(df.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}. Set PATIENT_COL and text columns correctly.")
df = df.dropna(subset=[PATIENT_COL, LABEL_COL]).copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)
if not set(df[LABEL_COL].unique()).issubset({0, 1}):
    raise ValueError("Outcome must be binary 0/1")
for col in set([TEXT_COL_EN, TEXT_COL_DANISH] + ALL_TEXT_COLS):
    df[col] = df[col].fillna("").astype(str)
df["all_texts"] = df[ALL_TEXT_COLS].agg(" ".join, axis=1).str.strip()
df = df.reset_index(drop=True)
print("Cases:", len(df), "Patients:", df[PATIENT_COL].nunique())


## 3. Fold-specific dataset tokenization

Tokenization uses the original checkpoint and text column for each experiment.


In [ ]:
def build_tokenized_dataset(model_ckpt, text_col, train_idx, val_idx=None, test_idx=None, max_length=128):
    tok = AutoTokenizer.from_pretrained(model_ckpt)
    def make(indices):
        frame = df.iloc[np.asarray(indices, dtype=int)]
        raw = Dataset.from_dict({"text": frame[text_col].tolist(), "labels": frame[LABEL_COL].astype(int).tolist()})
        encoded = raw.map(lambda batch: tok(batch["text"], truncation=True, padding="max_length", max_length=max_length),
                          batched=True, remove_columns=["text"])
        encoded.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
        return encoded
    datasets = {"train": make(train_idx)}
    if val_idx is not None: datasets["validation"] = make(val_idx)
    if test_idx is not None: datasets["test"] = make(test_idx)
    return datasets, tok


## 4. Shared Utilities

Defined once and reused by all five experiments.

In [ ]:
def compute_metrics(eval_pred):
    """Per-class P/R/F1, macro-F1, ROC-AUC, PR-AUC (threshold=0.5 during training only)."""
    logits, y = eval_pred
    logits = logits - logits.max(axis=1, keepdims=True)
    probs  = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
    y_prob = probs[:, 1]
    y_hat  = (y_prob >= 0.5).astype(int)

    prec, rec, f1, _ = precision_recall_fscore_support(y, y_hat, average=None, labels=[0, 1])
    macro_f1 = precision_recall_fscore_support(y, y_hat, average="macro")[2]
    return {
        "precision_0": prec[0], "recall_0": rec[0], "f1_0": f1[0],
        "precision_1": prec[1], "recall_1": rec[1], "f1_1": f1[1],
        "macro_f1":    macro_f1,
        "roc_auc":     roc_auc_score(y, y_prob),
        "pr_auc":      average_precision_score(y, y_prob),
    }

In [ ]:
class FocalLoss(nn.Module):
    """
    Multi-class focal loss (cross-entropy based).

    Args:
        alpha: optional per-class weight tensor of shape [num_classes]
        gamma: focusing parameter (default 2.0)
    """
    def __init__(self, alpha=None, gamma: float = 2.0, reduction: str = "mean"):
        super().__init__()
        self.alpha     = alpha
        self.gamma     = gamma
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        targets = targets.long()
        weight  = self.alpha.to(logits.device) if isinstance(self.alpha, torch.Tensor) else self.alpha
        ce      = F.cross_entropy(logits, targets, reduction="none")
        pt      = torch.exp(-ce)
        alpha_t = weight[targets] if weight is not None else 1.0
        loss    = alpha_t * ((1 - pt) ** self.gamma) * ce
        return loss.mean() if self.reduction == "mean" else loss.sum()


class FocalTrainer(Trainer):
    """HuggingFace Trainer subclass that replaces CE with Focal Loss."""
    def __init__(self, *args, focal_alpha=None, focal_gamma: float = 2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss = FocalLoss(alpha=focal_alpha, gamma=focal_gamma)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels").long()
        outputs = model(**inputs)
        loss    = self.focal_loss(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
def compute_class_weights(tokenized_dataset) -> torch.Tensor:
    """Inverse-frequency class weights from the training split (full, no sqrt softening)."""
    labels  = np.array(tokenized_dataset["train"]["labels"])
    counts  = np.bincount(labels, minlength=2)
    if (counts == 0).any():
        raise ValueError("Training fold contains only one class")
    weights = len(labels) / (2.0 * counts)  # full inverse-frequency weighting
    return torch.tensor(weights.astype("float32"), dtype=torch.float32)


def default_training_args(output_dir, num_epochs=5, evaluate=True):
    return TrainingArguments(
        output_dir=output_dir, learning_rate=1e-5,
        per_device_train_batch_size=16, per_device_eval_batch_size=16,
        num_train_epochs=num_epochs, weight_decay=0.01,
        eval_strategy="epoch" if evaluate else "no",
        save_strategy="epoch" if evaluate else "no",
        load_best_model_at_end=evaluate,
        metric_for_best_model="roc_auc" if evaluate else None,
        greater_is_better=True, save_total_limit=1,
        remove_unused_columns=False, report_to="none", seed=RANDOM_STATE,
    )


def probability_one(trainer, dataset):
    logits = trainer.predict(dataset).predictions
    logits = logits - logits.max(axis=1, keepdims=True)
    p = np.exp(logits)
    return p[:, 1] / p.sum(axis=1)


def best_threshold_from_probabilities(y, p):
    # Original notebook selected the threshold maximizing class-1 F1.
    precision, recall, thresholds = precision_recall_curve(y, p)
    if not len(thresholds): return 0.5
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    return float(thresholds[int(np.argmax(f1))])


def run_experiment(model_ckpt, text_col, output_dir, roc_save_path, max_length=128, seed=42):
    """Nested patient-grouped 5x5 CV. Inner CV selects epoch count and decision threshold.

    Each outer model is freshly initialized and trained on its entire outer training fold.
    The outer test fold is never used for model selection. Returns pooled out-of-fold results.
    """
    set_seed(seed)
    labels = df[LABEL_COL].to_numpy(dtype=int)
    groups = df[PATIENT_COL].to_numpy()
    outer = StratifiedGroupKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=seed)
    oof_prob = np.full(len(df), np.nan)
    oof_pred = np.full(len(df), -1, dtype=int)
    oof_fold = np.full(len(df), -1, dtype=int)
    thresholds_used, fold_results = [], []
    for outer_no, (train_idx, test_idx) in enumerate(outer.split(df, labels, groups), 1):
        if set(groups[train_idx]) & set(groups[test_idx]): raise AssertionError("Patient leakage in outer fold")
        inner = StratifiedGroupKFold(n_splits=INNER_FOLDS, shuffle=True, random_state=seed + outer_no)
        inner_y, inner_prob, best_epochs = [], [], []
        for inner_no, (itr, iva) in enumerate(inner.split(train_idx, labels[train_idx], groups[train_idx]), 1):
            actual_train, actual_val = train_idx[itr], train_idx[iva]
            if set(groups[actual_train]) & set(groups[actual_val]): raise AssertionError("Patient leakage in inner fold")
            ds, tok = build_tokenized_dataset(model_ckpt, text_col, actual_train, val_idx=actual_val, max_length=max_length)
            model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=2)
            args = default_training_args(f"{output_dir}/outer_{outer_no}/inner_{inner_no}", evaluate=True)
            trainer = FocalTrainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["validation"],
                                   data_collator=DataCollatorWithPadding(tokenizer=tok), compute_metrics=compute_metrics,
                                   focal_alpha=compute_class_weights(ds), focal_gamma=2.0)
            trainer.train()
            inner_prob.extend(probability_one(trainer, ds["validation"]).tolist())
            inner_y.extend(labels[actual_val].tolist())
            best_epochs.append(max(1, int(round(trainer.state.epoch or 5))))
            del trainer, model
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        chosen_epochs = max(1, int(np.median(best_epochs)))
        threshold = best_threshold_from_probabilities(np.array(inner_y), np.array(inner_prob))
        ds, tok = build_tokenized_dataset(model_ckpt, text_col, train_idx, test_idx=test_idx, max_length=max_length)
        model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=2)
        args = default_training_args(f"{output_dir}/outer_{outer_no}/final", num_epochs=chosen_epochs, evaluate=False)
        trainer = FocalTrainer(model=model, args=args, train_dataset=ds["train"],
                               data_collator=DataCollatorWithPadding(tokenizer=tok),
                               focal_alpha=compute_class_weights(ds), focal_gamma=2.0)
        trainer.train()
        probs = probability_one(trainer, ds["test"])
        oof_prob[test_idx] = probs
        oof_pred[test_idx] = (probs >= threshold).astype(int)
        oof_fold[test_idx] = outer_no
        thresholds_used.append(threshold)
        fold_results.append({"fold": outer_no, "threshold": threshold, "epochs": chosen_epochs,
                             "roc_auc": roc_auc_score(labels[test_idx], probs),
                             "macro_f1": precision_recall_fscore_support(labels[test_idx], oof_pred[test_idx], average="macro", zero_division=0)[2]})
        print(f"Outer fold {outer_no}/{OUTER_FOLDS}: AUC={fold_results[-1]['roc_auc']:.3f}, threshold={threshold:.3f}, epochs={chosen_epochs}")
        del trainer, model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    if np.isnan(oof_prob).any() or (oof_fold < 0).any(): raise AssertionError("Incomplete out-of-fold predictions")
    np.savez(roc_save_path, y_true=labels, y_prob1=oof_prob, y_pred=oof_pred,
             patient_id=groups.astype(str), fold=oof_fold, thresholds=np.asarray(thresholds_used))
    results = pd.DataFrame(fold_results)
    results.to_csv(roc_save_path.replace(".npz", "_fold_metrics.csv"), index=False)
    print(results.to_string(index=False))
    print("Mean ± SD outer-fold ROC-AUC:", results.roc_auc.mean(), results.roc_auc.std(ddof=1))
    print("Pooled OOF ROC-AUC:", roc_auc_score(labels, oof_prob))
    return results, oof_prob, labels, oof_pred, thresholds_used


## 5. DistilBERT — Prognosis Text

Input: translated prognosis note only (`prognosis_english`), `max_length=64`.

In [ ]:
fold_db, probs_db, labels_db, preds_db, thresh_db = run_experiment(
    model_ckpt   = "distilbert/distilbert-base-uncased",
    text_col     = TEXT_COL_EN,
    output_dir   = "results/distilbert_prognosis",
    roc_save_path= "rocdata_distilbert_prognosis_test.npz",
    max_length   = 64,
)

## 6. RoBERTa — Prognosis Text

Input: translated prognosis note only (`prognosis_english`), `max_length=128`.

In [ ]:
fold_rb, probs_rb, labels_rb, preds_rb, thresh_rb = run_experiment(
    model_ckpt   = "roberta-base",
    text_col     = TEXT_COL_EN,
    output_dir   = "results/roberta_prognosis",
    roc_save_path= "rocdata_roberta_prognosis_test.npz",
    max_length   = 128,
)

## 7. DistilBERT — All Texts

Input: prognosis note + signs & symptoms concatenated (`all_texts`), `max_length=128`.

In [ ]:
fold_db_all, probs_db_all, labels_db_all, preds_db_all, thresh_db_all = run_experiment(
    model_ckpt   = "distilbert/distilbert-base-uncased",
    text_col     = "all_texts",
    output_dir   = "results/distilbert_alltexts",
    roc_save_path= "rocdata_distilbert_alltexts_test.npz",
    max_length   = 128,
)

## 8. RoBERTa — All Texts

Input: prognosis note + signs & symptoms concatenated (`all_texts`), `max_length=128`.

In [ ]:
fold_rb_all, probs_rb_all, labels_rb_all, preds_rb_all, thresh_rb_all = run_experiment(
    model_ckpt   = "roberta-base",
    text_col     = "all_texts",
    output_dir   = "results/roberta_alltexts",
    roc_save_path= "rocdata_roberta_alltexts_test.npz",
    max_length   = 128,
)

## 9. Danish-BERT — Untranslated Danish Prognosis Text

Input: original untranslated Danish clinical text (`prognosis_danish`), `max_length=128`.

`Maltehb/danish-bert-botxo` is a BERT-base model pre-trained on Danish text. Using the untranslated source avoids any translation artefacts introduced by the English pipeline.

In [ ]:
fold_da, probs_da, labels_da, preds_da, thresh_da = run_experiment(
    model_ckpt   = "Maltehb/danish-bert-botxo",
    text_col     = TEXT_COL_DANISH,
    output_dir   = "results/danishbert_prognosis",
    roc_save_path= "rocdata_danishbert_prognosis_test.npz",
    max_length   = 128,
)

## 10. ROC Curve Visualisation

Pooled outer-fold out-of-fold ROC curves for all five models overlaid on a single publication-quality figure.
Edit `MODEL_FILES` to add or remove entries.

In [ ]:
MODEL_FILES = [
    ("DistilBERT (Prognosis)",   "rocdata_distilbert_prognosis_test.npz",  "#66c2a5"),
    ("RoBERTa (Prognosis)",      "rocdata_roberta_prognosis_test.npz",     "#fc8d62"),
    ("DistilBERT (All texts)",   "rocdata_distilbert_alltexts_test.npz",   "#8da0cb"),
    ("RoBERTa (All texts)",      "rocdata_roberta_alltexts_test.npz",      "#e78ac3"),
    ("Danish-BERT (All texts)",  "rocdata_danishbert_alltexts_test.npz",   "#a6d854"),
]

fig, ax = plt.subplots(figsize=(7, 6), dpi=150)

for label, path, color in MODEL_FILES:
    if not os.path.exists(path):
        print(f"Skipping {label}: {path} not found")
        continue
    d   = np.load(path)
    fpr, tpr, _ = roc_curve(d["y_true"], d["y_prob1"])
    auc = roc_auc_score(d["y_true"], d["y_prob1"])
    ax.plot(fpr, tpr, linewidth=2.5, color=color, label=f"{label}  AUC={auc:.2f}")

ax.plot([0, 1], [0, 1], "--", color="black", linewidth=1.5)
ax.set(xlabel="False Positive Rate", ylabel="True Positive Rate",
       title="ROC Curves – Pooled outer-fold predictions", xlim=(0, 1), ylim=(0, 1))
ax.spines[["top", "right"]].set_visible(False)
ax.legend(loc="lower right", frameon=False)
plt.tight_layout()
plt.savefig("roc_transformers_comparison.png", dpi=600, bbox_inches="tight")
plt.show()

## 11. Patient-level bootstrap confidence intervals

Bootstrap resamples patients, not individual teeth, from pooled outer-fold predictions. Classification metrics use each case’s own outer-fold threshold.


In [ ]:
def bootstrap_ci(y_true, y_prob, y_pred, patient_id, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    patient_id = np.asarray(patient_id).astype(str)
    unique_patients, inverse = np.unique(patient_id, return_inverse=True)
    patient_rows = [np.flatnonzero(inverse == i) for i in range(len(unique_patients))]
    metrics = {k: [] for k in ["roc_auc", "pr_auc", "f1_macro", "prec_c1", "rec_c1"]}
    for _ in range(n_boot):
        sampled = rng.integers(0, len(unique_patients), len(unique_patients))
        idx = np.concatenate([patient_rows[j] for j in sampled])
        yt, yp, yh = y_true[idx], y_prob[idx], y_pred[idx]
        if len(np.unique(yt)) < 2: continue
        metrics["roc_auc"].append(roc_auc_score(yt, yp))
        metrics["pr_auc"].append(average_precision_score(yt, yp))
        metrics["f1_macro"].append(precision_recall_fscore_support(yt, yh, average="macro", zero_division=0)[2])
        p, r, _, _ = precision_recall_fscore_support(yt, yh, average=None, labels=[0, 1], zero_division=0)
        metrics["prec_c1"].append(p[1]); metrics["rec_c1"].append(r[1])
    return {k: (float(np.mean(v)), float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))
            for k, v in metrics.items()} if metrics["roc_auc"] else {}

for name, path, _ in MODEL_FILES:
    if not os.path.exists(path):
        print(f"Skipping {name}"); continue
    with np.load(path) as data:
        y_true = data["y_true"].astype(int)
        y_prob = data["y_prob1"].astype(float)
        y_pred = data["y_pred"].astype(int)
        patient_id = data["patient_id"]
    stats = bootstrap_ci(y_true, y_prob, y_pred, patient_id, n_boot=BOOTSTRAP_SAMPLES)
    print(f"\n{name} (patient bootstrap)")
    for metric, (mean, lo, hi) in stats.items():
        print(f"  {metric:15s}: {mean:.3f} ({lo:.3f}–{hi:.3f})")
